# Reaction holdout evaluation

Evaluate P/N reaction predictions using the included holdout JSONL. Model calls are disabled initially.

## Inputs and setup

Install `python -m pip install -e ".[evaluation,notebook]"` from the repository root. The included data are ready for validation. Run the cells in order; leave both run switches `False` for local checks. All paths below are repository-relative.

| Input | Location | For another run |
| --- | --- | --- |
| Holdout records, JSONL | `data/final_json/holdout.jsonl` | Set `holdout_paths` in `configs/holdout_evaluation.json`; use message records with system/user inputs and an assistant `P` or `N` reference. |
| Training records, JSONL | `data/final_json/train.jsonl` | Set `training_path` to the corresponding training file for provenance. |
| Model settings, JSON | `configs/holdout_evaluation.json` | Select model IDs available to your API account and new output names for a fresh run. |
| Saved predictions, optional CSV | `results/evaluation/holdout/<output_name>.csv` | Set `PREDICTION_CSV` in the last section to analyze an existing file. |

For a small live test, set `settings["test_mode"] = True` and enable `RUN_EVALUATION`. `RUN_SANITY_TEST` makes a separate first-record request. Enter the API key only after enabling the relevant switch. Evaluation outputs are saved under `results/evaluation/holdout/`.

Implementation: [holdout evaluation and analysis](../src/mofinder/evaluation/holdout.py). See the [source-to-code guide](../docs/source_to_code.md) for the original workflow stages and their corresponding functions.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "mofinder").is_dir():
    raise FileNotFoundError("Open this notebook from the MOFinder repository or its notebooks folder.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from mofinder.display import display_paths
from mofinder.evaluation.holdout import (
    load_settings, validate_inputs, sanity_test, run_config, analyze_saved,
)

settings = load_settings(PROJECT_ROOT / "configs" / "holdout_evaluation.json")
RUN_SANITY_TEST = False
RUN_EVALUATION = False


## Inspect inputs and select models

The training file is recorded for provenance. Evaluation reads only the configured holdout records. Fine-tuned model IDs must be accessible to the account used for the run.


In [ ]:
validation = validate_inputs(settings)
display_paths(validation)


In [ ]:
# Use an output_name from settings["models"], or None to run both configurations.
MODEL_NAMES = None
settings["test_mode"] = False  # True evaluates at most ten pending records per model.
settings["models"]


## API key

Provide your OpenAI API key at the prompt after enabling a model call above. The key is kept in the current process and is not written to the notebook.


In [ ]:
import os
from getpass import getpass

if (RUN_SANITY_TEST or RUN_EVALUATION) and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key:")


## Optional first-record check

This separate call uses the configured first-record model and request settings. It reports the first output token and P/N log probabilities without writing prediction files.


In [ ]:
if RUN_SANITY_TEST:
    sanity_result = await sanity_test(settings)
    display(sanity_result)


## Evaluate the holdout

Only system and user messages are sent to the model. Assistant messages provide reference labels locally. Each completed request is saved; repeating a run skips every recorded attempt, including failed requests. Use a new output name to rerun them.


In [ ]:
if RUN_EVALUATION:
    evaluation_results = await run_config(settings, model_names=MODEL_NAMES)
    display(display_paths(evaluation_results))


## Analyze saved predictions

Set `PREDICTION_CSV` to a completed prediction file. Analysis requires no API key. Metrics use records with valid labels. Response coverage and failed records are reported alongside them.


In [ ]:
PREDICTION_CSV = None  # Example: settings["output_dir"] / "holdout_mofinder.csv"
if PREDICTION_CSV is not None:
    analysis = analyze_saved(PREDICTION_CSV)
    display(display_paths(analysis))
